In [1]:
import warnings
from langchain._api import LangChainDeprecationWarning

warnings.simplefilter("ignore", category=LangChainDeprecationWarning)

In [2]:
import os
from dotenv import load_dotenv,find_dotenv
_ = load_dotenv(find_dotenv())

groq_api_key =  os.environ["GROQ_API_KEY"]

In [3]:
from langchain_groq import ChatGroq
llmModel = ChatGroq(model = "llama3-70b-8192")

In [4]:
from typing import Optional
from langchain_core.pydantic_v1 import BaseModel,Field

class Person(BaseModel):
    """
    Notes:
    Optional[str]=>This means the value of name can either be a string (str) or None.
                   In other words, the field is not required—it's optional.
    Field(None, description=...)=>1. Sets the default value to None.
    2. Adds a description to help tools (like documentation generators or AI models) understand what this field represents.
    
    # Having a good description can help improve extraction results.
    """
    name: Optional[str]=Field(None, description="Name of the person")

    lastname: Optional[str]=Field(None, description="Last Name of the person")

    country: Optional[str]=Field(None, description="Country of the Person")

# Define the extractor

In [5]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt=ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert extraction algorithm. "
            "Only extract relevant information from the text. "
            "If you do not know the value of an attribute asked to extract, "
            "return null for the attribute's value.",
        ),
        ("human","{text}"),
    ]
)

In [6]:
chain = prompt | llmModel.with_structured_output(schema=Person)

# Testing

In [9]:
comment = "I loved the product. I have been using it from very long time. Its working perfectly fine.- Suyesh Prasad Rimal, Nepal"

In [10]:

chain.invoke({"text": comment})

Person(name='Suyesh Prasad', lastname='Rimal', country='Nepal')

# Extraction of a list of entities rather than a single entity

In [12]:
class Data(BaseModel):
    people: list[Person]

In [14]:
chain = prompt | llmModel.with_structured_output(schema=Data)

In [15]:
comment = "I'm so impressed with this product! It has truly transformed how I approach my daily tasks. The quality exceeds my expectations, and the customer support is truly exceptional. I've already suggested it to all my colleagues and relatives. - Emily Clarke, Canada"

In [16]:

chain.invoke({"text": comment})

Data(people=[Person(name='Emily', lastname='Clarke', country='Canada')])

# Several Reviews

In [17]:
text_input = """
Alice Johnson from Canada recently reviewed a book she loved. Meanwhile, Bob Smith from the USA shared his insights on the same book in a different review. Both reviews were very insightful.
"""

response = chain.invoke({"text": text_input})
response

Data(people=[Person(name='Alice', lastname='Johnson', country='Canada'), Person(name='Bob', lastname='Smith', country='USA')])